# Rare Disease, Real Kid — MVA Hackathon 2026 · Track 1

**Candidate variant identification in a proband with Mosaic Variegated Aneuploidy**

Author: Fernando (HF: `fernandosr85`) · Independent Researcher, Sao Paulo, Brazil  
Reference build: **GRCh38** · Submission window: 24 Aug – 24 Oct 2026

---

## Objective

Identify the variant pair most likely to explain the proband's MVA phenotype from
single-sample WGS while making the boundary between **established evidence** and
**unresolved inference** explicit.

The clinical label is given, so the task is not disease discovery. It is locus and
allele discrimination: MVA is genetically heterogeneous and recessive forms require
biallelic disease-causing variation.

## Audited result summary

**BUB1B remains the lead locus. One allele is solidly established; the second allele
and phase are unresolved.**

- `c.2210T>G` `p.Leu737Ter` — nonsense. Independent interpretation supports PVS1.
  Direct NCBI ClinVar verification (2026-09-02) identifies the same GRCh38 variant
  (`chr15:40209701 T>G`, rs759242053, Variation ID 533901) as **Pathogenic for MVA1**.
  The ClinVar submitter also warns that population-frequency data at this position
  are unreliable, so gnomAD frequency is not used as load-bearing PM2 evidence here.
- Competing second-allele hypotheses:
  - `c.3006T>G` `p.Asn1002Lys` — missense; REVEL 0.472 is indeterminate under the
    Pejaver calibration. No claim about NMD or hypomorphism is made.
  - `c.2679-1026A>G` — deep-intronic candidate. A splice effect is a hypothesis,
    not a demonstrated consequence; the public Ensembl REST VEP endpoint used here
    does not expose the SpliceAI plugin for this request (`invalid_field`).

The two coding BUB1B variants are ~10.9 kb apart and do not share a read-backed
phase block. A biallelic/compound-heterozygous genotype is therefore a **candidate
interpretation, not a demonstrated phase result**.

### What removing the dbSNP gate surfaced

Re-running the shortlist on all 61 coding/splice calls rather than the DB-filtered
four brought up **MAD1L1 `p.Arg59Cys`** (rs121908982), in MVA7 — an established MVA
locus. It entered on an aggregate `CLIN_SIG` pathogenic token, not on rarity.

**Two independent reasons remove it, and the order matters.**

*Population frequency, which is what the algorithm actually rejected it on.* At a
gnomAD frequency of 0.46%, Hardy-Weinberg predicts on the order of 10^5 homozygotes
worldwide against a stated MVA prevalence ceiling of fewer than 50 cases. No fully
penetrant recessive allele for this disease can be that common, whatever a database
asserts. Its REVEL of 0.267 reaches only BP4_Supporting, which on its own would not
have excluded it.

*Clinical context, which the automated field could not convey.* Direct review shows
the Pathogenic assertion on this allele is recorded in OMIM 602686 as allelic variant
.0002, **"PROSTATE CANCER, SOMATIC"** (Tsukasaki et al. 2001). The MAD1L1 alleles
reported for MVA7 are different variants (.0003 Gln66Ter, .0004 Glu628Ter). VEP's
`clin_sig` is allele-specific but neither condition-specific nor origin-specific, so
a somatic cancer submission is indistinguishable in that field from a germline
assertion for the disease under study.

That is a real limitation of automated annotation, and the pipeline was changed
because of it: a pathogenic token now **flags a variant for review** and records its
entry route. It cannot rescue a variant on its own. Confirmation of germline status
and a relevant condition is a recorded human decision.

There is also no second MAD1L1 allele, so no biallelic genotype exists to evaluate.

## Audit corrections implemented in this notebook

1. **dbSNP membership is no longer a rarity filter.** `DB` is retained only as QC
   metadata. Rarity filtering is based on gnomAD frequency returned by VEP.
2. **`check_existing` is enabled in VEP** so rsIDs and clinical significance from
   co-located known variants are captured in the same auditable annotation step.
3. **Chromosome-complement QC is explicit.** X/Y call patterns are checked before
   interpreting X-linked genes such as STAG2.
4. **No gene is automatically labelled a mapping artefact from het/hom ratio alone.**
   KNL1 remains an unresolved anomalous profile unless locus-level evidence proves
   ROH, paralogy or mapping failure.
5. **Allele-balance testing is worded correctly.** Failure to reject VAF=0.5 means
   compatibility with constitutional heterozygosity, not proof of germline origin.
6. **An aggregate `CLIN_SIG` token flags, it does not rescue.** The field is not
   condition- or origin-specific, so a pathogenic assertion admits a variant for
   review with its entry route recorded, and germline plus relevant-condition
   confirmation is required before it counts as support.
7. **Homozygosity blocks are tested with local depth, not gene-level averages.** A
   deletion confined to part of a gene is invisible in a gene-wide mean. Depth
   inside each block is compared with its immediate flanks, and no autozygosity
   mechanism is named for blocks far below conventional ROH segment lengths.
8. **Surviving candidates receive an explicit disposition.** The clinical override
   is permissive by design, so the shortlist is triaged rather than treated as a
   candidate list. Both disqualifying tests are derived: population frequency from
   Hardy-Weinberg against the stated MVA prevalence ceiling, and benign
   computational evidence from the Pejaver ladder, which is defined once and
   consulted by both triage and ACMG interpretation.
9. **A small KEEP-list annotation artefact is written** (`annotated_candidates.csv`)
   without genotype tables, preserving auditability while avoiding redistribution
   of the DELETE-list per-variant dataset.

## Data-handling compliance

Everything derived from the child's genome that contains broad per-variant genotype
information remains under `/kaggle/working/data` and is DELETE-list material. The
public repo should contain code, report, ranked findings and the small named-variant
annotation table only; notebook outputs remain disabled to avoid redistributing
large genotype tables.


## 0 · Environment and credentials

`HF_TOKEN` is read from Kaggle Secrets (Add-ons -> Secrets). Never paste a token
into a cell: cell source is saved with the notebook. Enable Internet in Settings.

In [ ]:
!pip install -q huggingface_hub pysam python-docx requests scipy

import os
from kaggle_secrets import UserSecretsClient

# Load the HF token from Kaggle Secrets into the environment variable that
# huggingface_hub reads. Printing only the boolean keeps the token out of the output.
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("token loaded:", bool(os.environ.get("HF_TOKEN")))

## 1 · Ingest

The full release is ~85 GB across 11 files (VCF + 8 FASTQ lanes + phenotype +
README). Track 1 needs 318 MB of it. There is **no BAM/CRAM** in the release, so
coordinate-range access to aligned reads is not possible without realignment —
which this analysis does not require.

In [ ]:
from huggingface_hub import hf_hub_download

REPO = "SageBio/mva-hackathon-2026-data"
DEST = "/kaggle/working/data"          # single directory -> single deletion at the end
os.makedirs(DEST, exist_ok=True)

# Only the files Track 1 needs. The 8 FASTQ lanes (~85 GB) are deliberately skipped.
for fname in [
    "WGS_EX2312012_HGWCNDSX7.vcf.gz",
    "WGS_EX2312012_HGWCNDSX7.vcf.gz.tbi",
    "Challenge_Clinical_Phenotype_1.docx",
    "README.md",
]:
    p = hf_hub_download(repo_id=REPO, filename=fname, repo_type="dataset",
                        local_dir=DEST)
    print(f"{os.path.getsize(p)/1e6:10.2f} MB  {p}")

VCF = DEST + "/WGS_EX2312012_HGWCNDSX7.vcf.gz"

## 2 · Header audit

Establishes three facts that constrain everything downstream:

1. **Reference build.** Confirmed GRCh38, and the header names the exact FASTA:
   `GCA_000001405.15_GRCh38_no_alt_analysis_set_plus_hs38d1_maskedGRC_exclusions_v2_no_chr.fasta`
   — no-alt analysis set with hs38d1 decoy, **no `chr` prefix**, `M` for
   mitochondria. Contig naming style must match in every fetch.
2. **The VCF is unannotated.** INFO holds caller metrics only (AC/AF/AN/DP/QD/FS/
   SOR/MQ/rank-sums). No CSQ, ANN, gnomAD or CLNSIG. **`AF` here is the
   within-sample allele fraction with AN=2 — always 0.5 or 1.0, never a population
   frequency.** Filtering on `AF < 0.01` would remove nothing and would fail silently.
3. **`PGT`/`PID` are present** — read-backed physical phasing. With a single
   sample and no trio, this is the only route to cis/trans, which a recessive
   diagnosis depends on.

In [ ]:
import pysam

vcf = pysam.VariantFile(VCF)
SAMPLE = list(vcf.header.samples)[0]
print("SAMPLE:", SAMPLE)

print("\nREFERENCE / PIPELINE:")
for line in str(vcf.header).split("\n"):
    if line.startswith(("##reference", "##source")) or "Sentieon" in line[:30]:
        print("  ", line[:170])

contigs = list(vcf.header.contigs)
print("\nCONTIG STYLE:", contigs[:3], "...", f"({len(contigs)} contigs)")
print("\nINFO  :", sorted(vcf.header.info.keys()))
print("FORMAT:", sorted(vcf.header.formats.keys()))

## 3 · Phenotype

HPO terms are explicitly in the KEEP list of the data-use terms, so they may be
retained and published. The source `.docx` is marked non-redistributable — extract
terms, do not reproduce the document.

Two features drive the interpretation:

- **Rhabdomyosarcoma + severe IUGR + short stature + failure to thrive** is the
  MVA1/BUB1B core. Note what is *absent*: **microcephaly**, one of the most
  consistent MVA1 features, is not listed — treated here as data, not oversight.
- **Recurrent parental miscarriage** is flagged by the organisers as phenotypic
  input rather than background. Partial spindle-checkpoint deficiency in a
  heterozygous carrier produces aneuploid embryos, so one mechanism accounts for
  both the proband and the reproductive history.

In [ ]:
from docx import Document
import re

doc = Document(f"{DEST}/Challenge_Clinical_Phenotype_1.docx")

# Extract only HPO identifiers and labels; the narrative text stays out of any
# artefact that will be published.
hpo = []
for t in doc.tables:
    for row in t.rows:
        cells_ = [c.text.strip() for c in row.cells]
        joined = " | ".join(cells_)
        m = re.search(r"(HP:\d{7})", joined)
        if m:
            hpo.append((m.group(1), cells_[1] if len(cells_) > 1 else cells_[0]))

print("HPO TERMS (KEEP-list, publishable):")
for hp_id, label in hpo:
    print(f"  {hp_id}  {label}")
print(f"\nn = {len(hpo)}")

## 4 · Panel definition and extraction

22 genes: the four established MVA loci plus the wider spindle-assembly-checkpoint
and chromosome-cohesion machinery. Coordinates are resolved live from Ensembl —
`assembly_name` is asserted to be GRCh38 so a build mismatch fails loudly rather
than producing an empty fetch.

`PAD = 5000` extends each gene span to catch promoter, UTR and near-splice variants.

In [ ]:
import requests, json, time
import pandas as pd, numpy as np

PANEL = [
    # Established MVA loci
    "BUB1B",    # MVA1  (MIM 257300)
    "CEP57",    # MVA2  (MIM 614114)
    "TRIP13",   # MVA3  (MIM 617598)
    "MAD1L1",   # MVA7  (MIM 620189)
    # Wider spindle-assembly checkpoint / segregation / cohesion
    "BUB1", "BUB3", "MAD2L1", "TTK", "CENPE", "CENPF",
    "KNL1", "NDC80", "ZW10", "ZWILCH", "CDC20", "AURKB",
    "ESPL1", "PTTG1", "PLK4", "SGO1", "STAG2", "RAD21",
]

def gene_structure(sym, retries=6):
    """Fetch gene span plus canonical-transcript exon structure from Ensembl.

    Only the gene symbol leaves this machine; no patient data is transmitted.

    Retries with exponential backoff and honours Retry-After on 429. A transient
    API failure here silently shrinks the panel, which changes every downstream
    count without changing anything visible in the output - so the caller asserts
    completeness rather than proceeding with whatever resolved.
    """
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(
                f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{sym}?expand=1",
                headers={"Content-Type": "application/json"}, timeout=60)
        except requests.RequestException as e:
            last = f"{type(e).__name__}: {e}"
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 200:
            break
        last = f"HTTP {r.status_code}"
        wait = float(r.headers.get("Retry-After", 0)) or (2 ** attempt)
        time.sleep(wait)
    else:
        print(f"  !! {sym}: unresolved after {retries} attempts ({last})")
        return None
    d = r.json()
    txs = d.get("Transcript", [])
    if not txs:
        return None
    canon = next((t for t in txs if t.get("is_canonical")), txs[0])
    tr = canon.get("Translation") or {}
    return {
        "gene": sym,
        "chrom": str(d["seq_region_name"]),         # no 'chr' prefix, matches VCF
        "start": int(d["start"]), "end": int(d["end"]),
        "length_kb": (int(d["end"]) - int(d["start"])) / 1000,
        "strand": d["strand"],
        "assembly": d.get("assembly_name"),
        "transcript": canon["id"],
        "exons": [(e["start"], e["end"]) for e in canon.get("Exon", [])],
        "cds_start": tr.get("start"), "cds_end": tr.get("end"),
    }

# A partial panel is a SILENT failure mode: the analysis completes, every count
# changes, and nothing in the output says a gene was dropped. In the run that
# exposed this, two genes failed to resolve and the notebook reported success.
# Panel completeness is therefore asserted, not hoped for.
REQUIRE_COMPLETE_PANEL = True

struct = {}
for s in PANEL:
    g = gene_structure(s)
    if g:
        struct[s] = g
    time.sleep(0.35)                                 # stay under Ensembl rate limit

PANEL_MISSING = [s for s in PANEL if s not in struct]

assert struct, "no gene structures resolved"
assert all(g["assembly"] == "GRCh38" for g in struct.values()), "unexpected assembly"

if PANEL_MISSING:
    msg = (f"PANEL INCOMPLETE: {len(PANEL_MISSING)} of {len(PANEL)} genes did not "
           f"resolve: {', '.join(PANEL_MISSING)}. Every downstream count is affected "
           f"and a causal variant in these genes could not be seen. Re-run this cell; "
           f"the failure is usually transient Ensembl rate limiting.")
    if REQUIRE_COMPLETE_PANEL:
        raise RuntimeError(msg)
    print("WARNING:", msg)

print(f"gene structures resolved: {len(struct)}/{len(PANEL)} (all GRCh38)")
assert len(struct) == len(PANEL) or not REQUIRE_COMPLETE_PANEL

In [ ]:
PAD = 5000          # promoter / UTR / near-splice margin
SPLICE_PAD = 8      # canonical splice site plus adjacent bases

def classify(gene, pos):
    """Position class relative to the canonical transcript of `gene`."""
    g = struct.get(gene)
    if not g:
        return "unknown"
    for (es, ee) in g["exons"]:
        if es <= pos <= ee:
            return "exonic"
        if es - SPLICE_PAD <= pos < es or ee < pos <= ee + SPLICE_PAD:
            return "splice_region"
    return "intronic" if g["start"] <= pos <= g["end"] else "flanking"

vcf = pysam.VariantFile(VCF)
rows = []
for g in struct.values():
    for rec in vcf.fetch(g["chrom"], max(0, g["start"] - PAD), g["end"] + PAD):
        s = rec.samples[SAMPLE]
        gt = s.get("GT")
        if gt is None or all(a in (0, None) for a in gt):
            continue
        ad = s.get("AD") or ()
        dp = s.get("DP")
        alt_ad = sum(ad[1:]) if len(ad) > 1 else None
        is_hom = gt[0] == gt[1] and gt[0] not in (0, None)
        rows.append({
            "gene": g["gene"], "chrom": rec.chrom, "pos": rec.pos,
            "ref": rec.ref, "alt": ",".join(rec.alts or []),
            "vfilter": ";".join(rec.filter.keys()) or "PASS",
            "gt": "/".join("." if a is None else str(a) for a in gt),
            "zyg": "hom" if is_hom else "het",
            "dp": dp, "gq": s.get("GQ"),
            "vaf": round(alt_ad / dp, 3) if alt_ad is not None and dp else None,
            "pid": s.get("PID"), "pgt": s.get("PGT"),
            "qd": rec.info.get("QD"),
            # DB is header/caller metadata only. It is NOT a rarity gate.
            # The lead pathogenic BUB1B variant has rs759242053 despite DB=False here.
            "vcf_db_flag": bool(rec.info.get("DB", False)),
            "region": classify(g["gene"], rec.pos),
        })

var = pd.DataFrame(rows)
var.to_csv(f"{DEST}/panel_variants.tsv", sep="\t", index=False)  # DELETE-list
print(f"non-reference calls across panel: {len(var)}")


## 5 · Per-gene triage and chromosome-complement QC

Per-gene density, heterozygosity fraction and depth are **diagnostics**, not automatic
exclusion rules. A low het ratio can reflect ROH, hemizygosity, caller representation,
paralogy or mapping problems. Those mechanisms must be separated before assigning a
label.

This matters for **STAG2**, which is X-linked. The previous notebook incorrectly
called its low heterozygosity a mapping artefact before determining the proband's
chromosome-complement pattern.

KNL1 is handled the same way. Its low heterozygosity is carried forward as a
measurement, and section 6c-bis tests the block's local depth to distinguish
copy-neutral homozygosity from loss of one haplotype. No mechanism is named before
that test runs.

The chromosome QC below uses only the VCF call pattern (not identity or gender):
X heterozygosity, X-to-autosome DP among called variants, and the number of PASS
non-reference calls on Y. The result is described as **XY-like / XX-like /
indeterminate**, not as a clinical sex determination.


In [ ]:
pas = var[var["vfilter"] == "PASS"]

summary = []
for s, g in struct.items():
    sub = pas[pas.gene == s]
    span_kb = g["length_kb"] + 2 * PAD / 1000
    n_het = int((sub.zyg == "het").sum())
    n_hom = int((sub.zyg == "hom").sum())
    summary.append({
        "gene": s, "kb": round(span_kb, 1), "n": len(sub),
        "per_kb": round(len(sub) / span_kb, 2),
        "het": n_het, "hom": n_hom,
        "het_ratio": round(n_het / max(1, n_het + n_hom), 3),
        "mean_dp": round(sub.dp.mean(), 1) if len(sub) else None,
        "exonic": int((sub.region == "exonic").sum()),
        "splice": int((sub.region == "splice_region").sum()),
        "db_flag_false": int((~sub.vcf_db_flag).sum()),
    })

sm = pd.DataFrame(summary).sort_values("per_kb")
print("=== PER-GENE DIAGNOSTIC PROFILE (no automatic exclusions) ===")
print(sm.to_string(index=False))


def chromosome_call_stats(chrom):
    """Summarise PASS non-reference calls on one chromosome from the VCF."""
    v = pysam.VariantFile(VCF)
    n = het = hom_alt = 0
    dps = []
    for rec in v.fetch(chrom):
        if "PASS" not in rec.filter.keys():
            continue
        s = rec.samples[SAMPLE]
        gt = s.get("GT")
        if gt is None or all(a in (0, None) for a in gt):
            continue
        n += 1
        if len(gt) >= 2 and gt[0] is not None and gt[1] is not None:
            if gt[0] != gt[1]:
                het += 1
            elif gt[0] not in (0, None):
                hom_alt += 1
        if s.get("DP") is not None:
            dps.append(s.get("DP"))
    return {
        "chrom": chrom,
        "n_pass_nonref": n,
        "het": het,
        "hom_alt": hom_alt,
        "het_fraction": het / max(1, het + hom_alt),
        "median_dp_at_calls": float(np.median(dps)) if dps else np.nan,
    }

chr_qc = pd.DataFrame([chromosome_call_stats(c) for c in ["15", "X", "Y"]])
print("\n=== CHROMOSOME-COMPLEMENT QC ===")
print(chr_qc.to_string(index=False))

auto_dp = chr_qc.loc[chr_qc.chrom == "15", "median_dp_at_calls"].iloc[0]
x_dp = chr_qc.loc[chr_qc.chrom == "X", "median_dp_at_calls"].iloc[0]
y_n = int(chr_qc.loc[chr_qc.chrom == "Y", "n_pass_nonref"].iloc[0])
x_het = chr_qc.loc[chr_qc.chrom == "X", "het_fraction"].iloc[0]
x_ratio = x_dp / auto_dp if auto_dp else np.nan

if 0.25 <= x_ratio <= 0.75 and y_n >= 100 and x_het < 0.20:
    chromosomal_pattern = "XY-like"
elif x_ratio >= 0.75 and y_n < 100:
    chromosomal_pattern = "XX-like"
else:
    chromosomal_pattern = "indeterminate"

print(f"\nVCF chromosome pattern: {chromosomal_pattern}")
print(f"X/autosome DP ratio at called sites: {x_ratio:.3f}")

print("\nInterpretation:")
print("  STAG2 low heterozygosity is not used as a mapping-artifact criterion.")
print("  KNL1 low heterozygosity is carried forward as a measurement; its copy")
print("  number is resolved by the local-depth test in section 6c-bis.")

# An XY complement has a downstream consequence that must be stated rather than
# left implicit: STAG2 is X-linked, so in an XY proband it is hemizygous. Any
# loss-of-function allele there would be unbuffered by a second copy, which makes
# STAG2 coding variants MORE consequential to check, not less. The retraction of
# the "mapping artefact" label therefore does not retire the locus - it has to be
# cleared on its own evidence.
X_LINKED_PANEL_GENES = ["STAG2"]
if chromosomal_pattern == "XY-like":
    print("\n  XY-like pattern - hemizygosity check for X-linked panel genes:")
    for gname in X_LINKED_PANEL_GENES:
        gsub = pas[pas.gene == gname]
        n_coding = int(gsub.region.isin(["exonic", "splice_region"]).sum())
        print(f"    {gname}: {n_coding} PASS coding/splice call(s) in the panel extraction")
        if n_coding == 0:
            print(f"      -> no coding candidate to follow up; recorded as a stated "
                  f"negative, not an assumption")
        else:
            print(f"      -> hemizygous: any LoF here would be unbuffered. "
                  f"Carried into the shortlist for annotation.")


## 6 · Population-frequency-first shortlist and annotation

The previous design used `DB=False` as a proxy for rarity. That is invalid: dbSNP
membership is not population frequency, and the lead `BUB1B p.Leu737Ter` variant is
rs759242053 even though this VCF carries `DB=False` at the site. A correctly
populated dbSNP flag would therefore have removed the answer.

The corrected design is:

1. take all PASS exonic / canonical splice-region calls in the panel;
2. annotate them with VEP;
3. request gnomAD frequency plus `check_existing` metadata;
4. apply a population-frequency gate **after** annotation;
5. retain a ClinVar Pathogenic/Likely Pathogenic override so a known disease variant
   is never dropped merely because a population field is noisy.

`DB` remains in the DELETE-list genotype table only as a diagnostic field. It never
participates in candidate selection.


In [ ]:
MAX_GNOMAD_AF = 1e-3   # conservative rare-disease screening gate; easy to audit/tune

# No dbSNP filter and no automatic artefact-gene exclusion.
coding_pool = pas[pas.region.isin(["exonic", "splice_region"])].copy()
print(f"coding/splice pool before population annotation: {len(coding_pool)}")


def parse_revel(raw):
    """Collapse dbNSFP's per-transcript REVEL string to one numeric score."""
    if raw is None:
        return None
    vals = {float(x) for x in str(raw).split(",") if x not in (".", "")}
    return max(vals) if vals else None


def vep(records, **opts):
    """Annotate a small variant set through Ensembl VEP REST.

    check_existing=1 adds co-located known-variant IDs and clinical significance.
    The public REST endpoint used here does not expose every VEP plugin; SpliceAI
    requests returned `invalid_field`, so no splice criterion is inferred from its
    absence.
    """
    payload = {
        "variants": [f"{r.chrom} {r.pos} . {r.ref} {r.alt}" for r in records],
        "canonical": 1,
        "hgvs": 1,
        "numbers": 1,
        "af_gnomadg": 1,
        "af_gnomade": 1,
        "check_existing": 1,
        "clin_sig_allele": 1,
        "dbNSFP": "REVEL_score",
    }
    payload.update(opts)
    r = requests.post(
        "https://rest.ensembl.org/vep/human/region",
        headers={"Content-Type": "application/json", "Accept": "application/json"},
        data=json.dumps(payload), timeout=180,
    )
    r.raise_for_status()
    return r.json()


def vep_chunked(frame, chunk=150, **opts):
    out = []
    recs = list(frame.itertuples())
    for i in range(0, len(recs), chunk):
        out.extend(vep(recs[i:i + chunk], **opts))
        time.sleep(0.2)
    return out


def input_key(v):
    p = str(v.get("input", "")).split()
    if len(p) < 5:
        return None
    return (str(p[0]), int(p[1]), str(p[3]), str(p[4]))


def vep_alt_alleles(v):
    """Return the alternate allele representation used by VEP for this input."""
    alts = {
        str(tc.get("variant_allele"))
        for tc in (v.get("transcript_consequences") or [])
        if tc.get("variant_allele") is not None
    }
    if not alts:
        allele_string = str(v.get("allele_string", ""))
        parts = allele_string.split("/")
        if len(parts) > 1:
            alts.update(parts[1:])
    return alts


def colocated_summary(v):
    """Allele-specific known-variant, clinical-significance and gnomAD summary."""
    rsids, clin, afs = set(), set(), []
    target_alts = vep_alt_alleles(v)
    for cv in v.get("colocated_variants") or []:
        vid = str(cv.get("id", ""))
        if vid.startswith("rs"):
            rsids.add(vid)
        for x in cv.get("clin_sig") or []:
            clin.add(str(x).lower())

        freqs = cv.get("frequencies") or {}
        matched = [(a, info) for a, info in freqs.items() if str(a) in target_alts]
        # Normalised indels can change allele representation (e.g. deletion -> '-').
        # If VEP reports exactly one frequency allele, it is safe to use as fallback
        # because check_existing is allele-specific by default. Never take the max
        # over unrelated alleles at a multiallelic locus.
        if not matched and len(freqs) == 1:
            matched = list(freqs.items())

        for _, allele_info in matched:
            for field in ("gnomadg", "gnomade"):
                val = allele_info.get(field)
                if val is not None:
                    try:
                        afs.append(float(val))
                    except (TypeError, ValueError):
                        pass
    return {
        "rsids": ";".join(sorted(rsids)),
        "clinvar_or_ensembl_clin_sig": ";".join(sorted(clin)),
        "gnomad_max_af": max(afs) if afs else np.nan,
    }

src_by_key = {
    (str(r.chrom), int(r.pos), str(r.ref), str(r.alt)): r
    for r in coding_pool.itertuples()
}

ann = vep_chunked(coding_pool)
annotation_rows = []
for v in ann:
    key = input_key(v)
    src = src_by_key.get(key)
    if src is None:
        continue
    tcs = v.get("transcript_consequences") or []
    tc = next((x for x in tcs if x.get("canonical") and x.get("gene_symbol") == src.gene),
              next((x for x in tcs if x.get("canonical")), tcs[0] if tcs else {}))
    col = colocated_summary(v)
    annotation_rows.append({
        "gene": src.gene,
        "chrom": str(src.chrom), "pos": int(src.pos),
        "ref": src.ref, "alt": src.alt,
        "consequence": ",".join(tc.get("consequence_terms", [])),
        "hgvsc": tc.get("hgvsc"), "hgvsp": tc.get("hgvsp"),
        "exon": tc.get("exon"), "intron": tc.get("intron"),
        "revel": parse_revel(tc.get("revel_score")),
        **col,
    })

ann_df = pd.DataFrame(annotation_rows)

PROTEIN_RELEVANT = {
    "stop_gained", "frameshift_variant", "missense_variant",
    "splice_acceptor_variant", "splice_donor_variant", "splice_region_variant",
    "start_lost", "stop_lost", "inframe_insertion", "inframe_deletion",
    "protein_altering_variant",
}

def relevant_consequence(s):
    return bool(set(str(s).split(",")) & PROTEIN_RELEVANT)

def clin_pathogenic(s):
    vals = set(str(s).split(";"))
    return bool(vals & {"pathogenic", "likely_pathogenic", "pathogenic/likely_pathogenic"})

# ---------------------------------------------------------------------------
# CLIN_SIG admits a variant for REVIEW. It does not validate one.
#
# VEP's clin_sig is aggregated per allele, but NOT per condition and NOT per
# origin. A single "pathogenic" token in that list can come from a somatic cancer
# submission for an unrelated disease and look identical to a germline assertion
# for the condition under study.
#
# That is exactly what happened here. MAD1L1 p.Arg59Cys (rs121908982) carries a
# Pathogenic assertion recorded in OMIM 602686 as allelic variant .0002,
# "PROSTATE CANCER, SOMATIC" (Tsukasaki et al. 2001). The MAD1L1 alleles reported
# for MVA7 are different variants entirely (.0003 Gln66Ter, .0004 Glu628Ter).
#
# A pathogenic token therefore FLAGS a variant into the pool with its entry route
# recorded. Triage then requires germline + relevant-condition confirmation before
# the flag can rescue anything. Nothing is auto-rescued by an aggregate label.
# ---------------------------------------------------------------------------
rare = ann_df.gnomad_max_af.isna() | (ann_df.gnomad_max_af <= MAX_GNOMAD_AF)
clin_flag = ann_df.clinvar_or_ensembl_clin_sig.map(clin_pathogenic)
impact = ann_df.consequence.map(relevant_consequence)

candidate_ann = ann_df[(rare | clin_flag) & impact].copy()
_r = rare.loc[candidate_ann.index]
_c = clin_flag.loc[candidate_ann.index]
candidate_ann["entry_route"] = np.where(_r & _c, "rarity+clinical_flag",
                               np.where(_r, "rarity", "clinical_flag_only"))

# Positions whose clinical assertion has been manually confirmed as GERMLINE and
# for a condition relevant to this case. Empty by design: an entry here is a
# recorded human decision, never an automated inference from an aggregate field.
CLINVAR_CONTEXT_VERIFIED = {
    # pos: "reason confirmed germline + relevant condition"
}

print("\n=== CORRECTED RARE/CLINICAL SHORTLIST ===")
cols = ["gene", "chrom", "pos", "consequence", "hgvsc", "hgvsp",
        "gnomad_max_af", "rsids", "clinvar_or_ensembl_clin_sig", "revel"]
print(candidate_ann[cols].sort_values(["gene", "pos"]).to_string(index=False))

# ---------------------------------------------------------------------------
# Why each surviving candidate is kept or rejected.
#
# The clinical override is deliberately permissive: it admits any variant with a
# Pathogenic/Likely Pathogenic assertion even when the frequency gate would have
# removed it. That is the right default for a screen, but it means the surviving
# set needs an explicit disposition rather than being treated as a candidate list.
#
# The decisive test for a fully penetrant recessive allele is prevalence
# arithmetic, not annotation. MVA affects fewer than 50 people worldwide. Under
# Hardy-Weinberg, an allele at frequency p predicts a homozygote frequency of p^2,
# so a common allele is incompatible with the disease however it is annotated.
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Pejaver et al. (2022) calibrated REVEL thresholds - full ladder.
#
# Defined here, BEFORE first use, so that candidate triage and ACMG interpretation
# consult one definition instead of duplicating cut-offs. Section 8a displays this
# same ladder; it does not redefine it.
#
# Pathogenic side: evidence strengthens as the score rises.
# Benign side: evidence strengthens as the score falls.
# ---------------------------------------------------------------------------
REVEL_PATHOGENIC = [           # (minimum score, criterion, ACMG strength key)
    (0.932, "PP3_Strong",     "PS"),
    (0.773, "PP3_Moderate",   "PM"),
    (0.644, "PP3_Supporting", "PP"),
]
REVEL_BENIGN = [               # (maximum score, criterion, ACMG strength key)
    (0.003, "BP4_VeryStrong", "BS"),   # capped at Strong in practice
    (0.016, "BP4_Strong",     "BS"),
    (0.053, "BP4_Moderate",   "BM"),
    (0.290, "BP4_Supporting", "BP"),
]
PP3_MIN = REVEL_PATHOGENIC[-1][0]      # 0.644 - weakest pathogenic evidence
BP4_MAX = REVEL_BENIGN[-1][0]          # 0.290 - weakest benign evidence

# Any benign criterion at Moderate strength or above is treated as disqualifying
# for a candidate. Derived from the ladder, never hard-coded.
BENIGN_DISQUALIFYING = {"BM", "BS", "BA"}


def revel_band(score):
    """Strongest ACMG criterion a REVEL score supports, or None.

    Returns (criterion, strength_key, explanation). Thresholds are walked from
    strongest to weakest on each side, so a score qualifies at the highest strength
    its value earns rather than defaulting to Supporting.
    """
    if score is None or (isinstance(score, float) and pd.isna(score)):
        return None, None, "no score (non-missense or unscored)"
    for cutoff, crit, strength in REVEL_PATHOGENIC:
        if score >= cutoff:
            return crit, strength, f"REVEL {score} >= {cutoff}"
    for cutoff, crit, strength in REVEL_BENIGN:
        if score <= cutoff:
            return crit, strength, f"REVEL {score} <= {cutoff}"
    return None, None, (f"INDETERMINATE ({BP4_MAX} < {score} < {PP3_MIN}) - "
                        "no criterion applies at any strength")


WORLD_POP = 8e9
MVA_WORLD_CASES = 50          # organiser-stated prevalence ceiling

def hw_incompatible(af):
    """Expected homozygotes worldwide under Hardy-Weinberg, vs MVA prevalence."""
    if pd.isna(af) or af <= 0:
        return False, np.nan, np.nan
    expected = (af ** 2) * WORLD_POP
    return expected > MVA_WORLD_CASES * 10, expected, expected / MVA_WORLD_CASES

def triage(r):
    """Disposition for one surviving candidate, with the reason recorded.

    Both disqualifying tests are derived, not hard-coded: population frequency from
    Hardy-Weinberg against the stated prevalence ceiling, and benign computational
    evidence from the Pejaver ladder defined above.
    """
    too_common, expected, factor = hw_incompatible(r.gnomad_max_af)
    if too_common:
        return ("rejected_population_frequency",
                f"gnomAD AF {r.gnomad_max_af:.2%} predicts ~{expected:,.0f} homozygotes "
                f"worldwide ({factor:,.0f}x the MVA prevalence ceiling); incompatible "
                f"with a fully penetrant recessive allele regardless of annotation")

    crit, strength, why = revel_band(r.revel)
    if strength in BENIGN_DISQUALIFYING:
        return ("deprioritised_benign_computational",
                f"{crit} on the Pejaver ladder ({why})")

    # A variant admitted ONLY by an aggregate clinical label has not been shown to
    # carry a germline assertion for a relevant condition. It is not treated as
    # supported until that context is confirmed by hand.
    if getattr(r, "entry_route", "") == "clinical_flag_only" \
            and r.pos not in CLINVAR_CONTEXT_VERIFIED:
        return ("requires_clinvar_context_review",
                "admitted only by an aggregate CLIN_SIG pathogenic token, which is "
                "neither condition- nor origin-specific; germline status and disease "
                "context must be confirmed directly in ClinVar before this counts "
                "as supporting evidence")

    if r.gene == "BUB1B":
        return ("retained_BUB1B_hypothesis", "lead locus; carried to interpretation")
    return ("review", "no disqualifying evidence; requires manual assessment")

triaged = candidate_ann.copy()
triaged[["audit_status", "audit_reason"]] = triaged.apply(
    lambda r: pd.Series(triage(r)), axis=1)

print("\n=== CANDIDATE TRIAGE ===")
for r in triaged.sort_values(["gene", "pos"]).itertuples():
    print(f"  {r.gene} {r.chrom}:{r.pos} {r.hgvsp or r.hgvsc}")
    print(f"    -> {r.audit_status}")
    print(f"       {r.audit_reason}")

# KEEP-list audit artefact: the CORRECTED shortlist with its disposition, plus any
# variant from the original DB-gated shortlist that no longer survives, so the
# artefact documents the pipeline that actually ran and what the fix changed.
HISTORICAL_DB_SHORTLIST = {40209701, 40220612, 123164968, 20174258}
extra = ann_df[ann_df.pos.isin(HISTORICAL_DB_SHORTLIST)
               & ~ann_df.pos.isin(triaged.pos)].copy()
if len(extra):
    extra["audit_status"] = "dropped_by_corrected_filter"
    extra["audit_reason"] = (
        "passed the old dbSNP-based novelty gate but fails the corrected "
        "population-frequency and protein-consequence filters")

audit_out = pd.concat([triaged, extra], ignore_index=True)
keep_cols = ["gene", "chrom", "pos", "ref", "alt", "consequence", "hgvsc", "hgvsp",
             "exon", "intron", "gnomad_max_af", "rsids",
             "clinvar_or_ensembl_clin_sig", "revel", "entry_route",
             "audit_status", "audit_reason"]
for _c in keep_cols:
    if _c not in audit_out.columns:
        audit_out[_c] = pd.NA
audit_path = "/kaggle/working/annotated_candidates.csv"
audit_out[keep_cols].sort_values(["gene", "pos"]).to_csv(audit_path, index=False)
print(f"\nKEEP-list annotation written: {audit_path} ({len(audit_out)} rows)")


### 6b · Deep-intronic scan in the lead gene

Once BUB1B is the lead locus, intronic variation is reviewed separately. The previous
text incorrectly described SpliceAI's `-D=50` setting as a ±50 bp distance from an
exon. That is not what `-D` means.

The actual limitation in this workflow is simpler: the public Ensembl REST endpoint
used for annotation does not provide the requested SpliceAI plugin field and returned
`invalid_field`. Therefore **no PP3/splice evidence is assigned** to
`c.2679-1026A>G`.

The variant remains a deep-intronic **splice-altering hypothesis**. Pseudo-exon
inclusion is biologically plausible in this mechanism class, but it is not established
for this nucleotide change without dedicated splice prediction and, preferably,
patient RNA or a minigene assay.


In [ ]:
LEAD_GENE = "BUB1B"

deep_pool = pas[(pas.gene == LEAD_GENE) & (pas.region == "intronic")].copy()
print(f"BUB1B intronic PASS calls reviewed: {len(deep_pool)}")

# dbSNP membership is deliberately NOT used here.
deep_ann_raw = vep_chunked(deep_pool, distance=5000)
deep_src = {
    (str(r.chrom), int(r.pos), str(r.ref), str(r.alt)): r
    for r in deep_pool.itertuples()
}

deep_rows = []
for v in deep_ann_raw:
    key = input_key(v)
    src = deep_src.get(key)
    if src is None:
        continue
    tcs = v.get("transcript_consequences") or []
    tc = next((x for x in tcs if x.get("canonical") and x.get("gene_symbol") == LEAD_GENE),
              next((x for x in tcs if x.get("canonical")), tcs[0] if tcs else {}))
    deep_rows.append({
        "gene": LEAD_GENE, "chrom": str(src.chrom), "pos": int(src.pos),
        "ref": src.ref, "alt": src.alt,
        "consequence": ",".join(tc.get("consequence_terms", [])),
        "hgvsc": tc.get("hgvsc"), "intron": tc.get("intron"),
        **colocated_summary(v),
    })

deep_ann_df = pd.DataFrame(deep_rows)
deep_rare = deep_ann_df[
    deep_ann_df.gnomad_max_af.isna() | (deep_ann_df.gnomad_max_af <= MAX_GNOMAD_AF)
].copy()

print("\n=== RARE/ABSENT BUB1B INTRONIC CALLS ===")
print(deep_rare.sort_values("pos").to_string(index=False))

# Named second-allele hypothesis: KEEP-list finding, not a broad genotype table.
target = deep_ann_df[deep_ann_df.pos == 40216470].copy()
if len(target):
    target["splice_evidence"] = (
        "not scored: SpliceAI plugin unavailable on the public REST request; "
        "pseudo-exon effect requires dedicated prediction and RNA/minigene validation"
    )
    path = "/kaggle/working/deep_intronic_candidate.csv"
    target.to_csv(path, index=False)
    print("\nKEEP-list deep-intronic finding written:", path)


### 6c · Intragenic LOH scan — a CNV proxy from data already in hand

No CNV/SV caller was run, so a large deletion as the second allele has not been
excluded. One informative check is available without new data.

A heterozygous deletion covering part of a gene removes one haplotype, so every
call inside the deleted interval is reported as apparently homozygous. That leaves
a signature: a **contiguous block of homozygous calls** bounded by heterozygous
calls on both sides. Its absence across a gene argues against a large intragenic
deletion.

The honest limit: this is a marker-density argument, not a depth-based CNV call.
With a handful of markers across tens of kb, a small single-exon deletion can sit
between two markers and leave no trace. It argues against a **broad** deletion,
and it does not exclude a focal one.

In [ ]:
def loh_scan(gene, frame):
    """Longest run of consecutive apparently-homozygous calls within a gene.

    A heterozygous deletion presents as a contiguous homozygous block flanked by
    heterozygous calls. This is a marker-based proxy, not a depth-based CNV call.
    """
    g = frame[frame.gene == gene].sort_values("pos")
    if g.empty:
        return None
    zyg = list(g.zyg)
    pos = list(g.pos)

    best_len, best_span, best_range = 0, 0, (None, None)
    run_start = None
    for i, z in enumerate(zyg + ["het"]):          # sentinel closes a trailing run
        if z == "hom":
            if run_start is None:
                run_start = i
        else:
            if run_start is not None:
                length = i - run_start
                span = pos[i - 1] - pos[run_start]
                if length > best_len or (length == best_len and span > best_span):
                    best_len, best_span = length, span
                    best_range = (pos[run_start], pos[i - 1])
                run_start = None

    gene_span = pos[-1] - pos[0] if len(pos) > 1 else 0
    return {
        "gene": gene,
        "n_markers": len(g),
        "n_het": int((g.zyg == "het").sum()),
        "n_hom": int((g.zyg == "hom").sum()),
        "longest_hom_run": best_len,
        "run_span_bp": best_span,
        "run_start": best_range[0], "run_end": best_range[1],
        "marker_span_bp": gene_span,
        "frac_span_in_run": round(best_span / gene_span, 3) if gene_span else 0.0,
        "median_gap_bp": int(np.median(np.diff(pos))) if len(pos) > 2 else None,
    }

loh = pd.DataFrame([r for r in (loh_scan(g, pas) for g in struct) if r])
loh = loh.sort_values("frac_span_in_run", ascending=False)

print("=== INTRAGENIC LOH SCAN (CNV proxy) ===")
print(loh.to_string(index=False))

lead = loh[loh.gene == LEAD_GENE]
if len(lead):
    r = lead.iloc[0]
    print(f"\n--- {LEAD_GENE} ---")
    print(f"  markers: {r.n_markers} ({r.n_het} het / {r.n_hom} hom) "
          f"across {r.marker_span_bp:,} bp")
    print(f"  longest homozygous run: {r.longest_hom_run} marker(s), "
          f"{r.run_span_bp:,} bp -> {r.frac_span_in_run:.1%} of the marker span")
    print(f"  median inter-marker gap: {r.median_gap_bp:,} bp"
          if r.median_gap_bp else "  median inter-marker gap: n/a")

    if r.frac_span_in_run < 0.30 and r.n_het >= 4:
        print("\n  READ: heterozygous calls are interleaved across the full span with "
              "no extended homozygous block.")
        print("  -> argues AGAINST a large heterozygous deletion spanning the gene "
              "as the second allele.")
        print(f"  -> does NOT exclude a focal deletion smaller than the "
              f"{r.median_gap_bp:,} bp median marker gap, nor a deletion outside "
              f"the called interval." if r.median_gap_bp else
              "  -> does NOT exclude a focal deletion between markers.")
    else:
        print("\n  READ: an extended homozygous block is present. This warrants "
              "depth-based CNV assessment before interpreting the genotype.")

print("\nScope note: this test uses called variants only. A dedicated CNV/SV "
      "caller on aligned reads remains the definitive assessment and was not run.")

### 6c-bis · Local depth inside homozygosity blocks — copy-neutral or deletion?

The LOH scan finds blocks of apparent homozygosity but cannot say what causes them.
Gene-level mean depth does not settle it either: a deletion confined to part of a
gene is diluted by the rest of the gene's coverage, so a normal gene-wide average is
compatible with a locally deleted segment.

The discriminating measurement is **local**. A heterozygous deletion removes one
haplotype inside the block, so depth at called sites there should fall to roughly
half the immediately flanking depth. Copy-neutral homozygosity — a homozygous
haplotype, a run of autozygosity, or an LD block — leaves depth unchanged.

**On calling these blocks "ROH":** conventional runs-of-homozygosity analysis uses
minimum segment lengths of several hundred kb to megabases, precisely so that short
haplotype and LD blocks are not mislabelled as autozygosity. A block of tens of kb is
far below that. It is reported here as a localized homozygosity signal with its
depth ratio, and no autozygosity mechanism is asserted.

In [ ]:
def local_depth_profile(chrom, start, end, flank_bp=50_000, min_sites=10):
    """Median depth at PASS calls inside a window versus its immediate flanks.

    Deletion of one haplotype removes half the reads: inside/flank ratio ~0.5.
    Copy-neutral homozygosity leaves coverage intact: ratio ~1.0.
    """
    v = pysam.VariantFile(VCF)

    def depths(a, b):
        out = []
        for rec in v.fetch(chrom, max(0, int(a)), int(b)):
            if "PASS" not in rec.filter.keys():
                continue
            dp = rec.samples[SAMPLE].get("DP")
            if dp is not None:
                out.append(dp)
        return out

    inside = depths(start, end)
    up     = depths(start - flank_bp, start)
    down   = depths(end, end + flank_bp)
    flank  = up + down

    if len(inside) < min_sites or len(flank) < min_sites:
        return None

    # NOTE on sex chromosomes: the 0.70/1.30 bands assume a diploid region. On a
    # hemizygous chromosome (X or Y in an XY sample) there is only one copy to
    # begin with, so a deletion drives depth toward zero rather than toward half.
    # A ratio near 1.0 there means the block is uniform within an already
    # single-copy chromosome; it is not evidence about "one haplotype".
    med_in, med_fl = float(np.median(inside)), float(np.median(flank))
    ratio = med_in / med_fl if med_fl else np.nan
    if ratio < 0.70:
        call = "DEPTH-REDUCED - consistent with loss of one haplotype (deletion)"
    elif ratio > 1.30:
        call = "DEPTH-ELEVATED - possible duplication or mismapping pile-up"
    else:
        call = "COPY-NEUTRAL - homozygosity without depth loss"
    return {"chrom": chrom, "start": int(start), "end": int(end),
            "n_inside": len(inside), "n_flank": len(flank),
            "median_dp_inside": med_in, "median_dp_flank": med_fl,
            "inside_over_flank": round(ratio, 3), "call": call}


# Blocks worth testing: the longest homozygous runs found by the LOH scan.
MIN_RUN_MARKERS = 10
blocks = loh[(loh.longest_hom_run >= MIN_RUN_MARKERS) & loh.run_start.notna()]

print("=== LOCAL DEPTH INSIDE HOMOZYGOSITY BLOCKS ===")
rows = []
for r in blocks.itertuples():
    chrom = struct[r.gene]["chrom"]
    prof = local_depth_profile(chrom, r.run_start, r.run_end)
    if prof is None:
        print(f"  {r.gene}: too few sites to test")
        continue
    prof["gene"] = r.gene
    prof["run_markers"] = r.longest_hom_run
    prof["run_kb"] = round(r.run_span_bp / 1000, 1)
    rows.append(prof)

if rows:
    dp_df = pd.DataFrame(rows)[
        ["gene", "chrom", "start", "end", "run_markers", "run_kb",
         "median_dp_inside", "median_dp_flank", "inside_over_flank", "call"]]
    print(dp_df.to_string(index=False))

    print("\nReading (diploid regions):")
    print("  ratio ~1.0  -> copy-neutral homozygosity (haplotype / LD / autozygosity)")
    print("  ratio ~0.5  -> one haplotype absent: heterozygous deletion")
    print("\nOn X/Y in an XY sample the region is already single-copy, so a deletion")
    print("drives depth toward zero, not toward half. A ratio near 1.0 there means")
    print("the block is uniform within a hemizygous chromosome.")
    print("\nSegment length matters for naming the mechanism. Conventional ROH")
    print("analysis uses minimum segments of several hundred kb to megabases, so")
    print("blocks of tens of kb are reported here as localized homozygosity")
    print("signals, not as runs of autozygosity.")
else:
    print("  no block met the site-count threshold")

# The lead gene's own coding calls: what was actually evaluated at anomalous loci.
for gname in blocks.gene:
    g = pas[pas.gene == gname]
    n_coding = int(g.region.isin(["exonic", "splice_region"]).sum())
    print(f"\n  {gname}: {n_coding} PASS coding/splice call(s) entered the corrected "
          f"annotation pool; survivors of the population/clinical shortlist: "
          f"{int((candidate_ann.gene == gname).sum())}")

### 6d · Dedicated splice prediction for the deep-intronic candidate — opt-in

The Ensembl REST endpoint does not expose the SpliceAI plugin, so section 6b
records the absence of a score rather than inferring anything from it. A dedicated
lookup service can supply one.

**This is off by default and requires a decision from the operator.** Sending a
patient variant coordinate to a third-party service is governed by the Processor /
Recipient test set out by the Sage Privacy Office: the service must not train on
the input, must not acquire rights over it, and must retain it only briefly and for
a limited purpose. Read the terms of whichever service you use, then set the flag
below and record the provider in the methods declaration — exactly as you would for
any other external tool.

If the flag stays `False`, the analysis proceeds with the splice question openly
unresolved, which is a defensible state and is what the current submission
describes.

In [ ]:
# Off by default. Set to True only after reading the service's data-handling terms
# and confirming it qualifies as a processor under the challenge rules.
ENABLE_EXTERNAL_SPLICE_API = False

SPLICE_TARGET = ("15", 40216470, "A", "G")   # BUB1B c.2679-1026A>G

def spliceai_lookup(chrom, pos, ref, alt, distance=500, mask=0, timeout=120):
    """Query a SpliceAI lookup service for delta scores around a variant.

    `distance` is the maximum distance between the variant and a gained/lost splice
    site - NOT a window measured from the exon. A deep-intronic variant can create a
    cryptic site near itself and be scored; widening `distance` extends the search
    for that site. Default 500 bp comfortably covers a pseudo-exon hypothesis.
    """
    url = ("https://spliceailookup-api.broadinstitute.org/spliceai/"
           f"?hg=38&distance={distance}&mask={mask}"
           f"&variant=chr{chrom}-{pos}-{ref}-{alt}")
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()

splice_result = None
if ENABLE_EXTERNAL_SPLICE_API:
    try:
        splice_result = spliceai_lookup(*SPLICE_TARGET)
        print("=== SPLICE PREDICTION ===")
        print(json.dumps(splice_result, indent=2)[:2000])
        print("\nRecord provider, plan and data-handling configuration in the "
              "methods declaration.")
    except Exception as e:
        print(f"splice lookup failed: {type(e).__name__}: {e}")
        print("Proceeding with the splice question unresolved.")
else:
    print("External splice prediction: DISABLED.")
    print()
    print("Current evidence state for BUB1B c.2679-1026A>G:")
    print("  - deep intronic, 1,026 bp from the intron 20 acceptor")
    print("  - rare/absent in gnomAD")
    print("  - no splice prediction obtained; no splice criterion assigned")
    print("  - pseudo-exon activation is a mechanistic hypothesis, untested here")
    print()
    print("To resolve: enable the flag above after a terms review, or run SpliceAI/"
          "Pangolin locally. Decisive evidence would be patient RNA-seq or a "
          "minigene assay, neither of which is available in this dataset.")

## 7 · Phase and allele-balance check

**Phase.** The two coding BUB1B candidates are ~10.9 kb apart and do not share a
`PID` phase block. Single-sample short reads therefore do not establish cis/trans.
A candidate biallelic interpretation is biologically coherent with recessive MVA1,
but **trans is not demonstrated**. Trio, long-read or linked-read data would resolve
this directly.

**Allele balance.** Candidate VAFs are tested against a binomial expectation of 0.5
and compared with high-quality heterozygous panel calls. A non-significant result is
reported only as **compatible with constitutional heterozygosity**. It does not prove
germline origin and it does not test the chromosomal mosaicism that defines MVA.


In [ ]:
from scipy import stats

b = pas[(pas.gene == LEAD_GENE) & (pas.pos.isin([40209701, 40220612]))]
print("=== PHASE ===")
print(b[["pos", "gt", "dp", "gq", "vaf", "pid", "pgt"]].to_string(index=False))
if len(b) >= 2:
    span = int(b.pos.max() - b.pos.min())
    shared = b.pid.notna().all() and b.pid.nunique() == 1
    print(f"\nseparation: {span:,} bp")
    print("phase:", "resolved by PGT" if shared else
          "UNRESOLVED - no shared read-backed phase block; trans not demonstrated")

het = pas[(pas.zyg == "het") & (pas.gq >= 90) & (pas.dp >= 20)]
print(f"\n=== ALLELE-BALANCE CHECK (n={len(het)} high-quality het calls) ===")
print(f"median VAF: {het.vaf.median():.3f} | 5th/95th pct: "
      f"{het.vaf.quantile(0.05):.3f}/{het.vaf.quantile(0.95):.3f}")

test_set = pas[(pas.gene == LEAD_GENE) & pas.pos.isin([40209701, 40220612, 40216470])]
for r in test_set.itertuples():
    alt_reads = round(r.vaf * r.dp)
    p = stats.binomtest(alt_reads, int(r.dp), 0.5).pvalue
    pct = (het.vaf < r.vaf).mean() * 100
    verdict = ("compatible with constitutional heterozygosity" if p > 0.01
               else "REVIEW - significant allele-balance deviation")
    print(f"  {r.pos}: {alt_reads}/{r.dp} alt | p={p:.3f} | pct {pct:.0f} | {verdict}")


## 8 · ACMG/AMP interpretation and external ClinVar evidence

The internal evidence assessment and the external database assertion are kept
separate.

### `BUB1B c.2210T>G p.Leu737Ter`

Independent sequence interpretation supports **PVS1**: this is a nonsense variant in
an established loss-of-function disease gene, sufficiently upstream for predicted
NMD. Under the Tavtigian point framework, PVS1 alone is 8 points → **Likely
Pathogenic**.

The previous notebook also added PM2 from a small gnomAD frequency. That is no longer
load-bearing because the current NCBI ClinVar record explicitly states that population
frequency data at this position are unreliable due to poor data quality.

Separately, direct NCBI ClinVar verification identifies this exact variant as
**Pathogenic for Mosaic Variegated Aneuploidy syndrome 1** (RCV000641226.9;
Variation ID 533901; rs759242053; one submitter with criteria provided). This external
assertion strengthens the etiologic case for BUB1B but is not double-counted as an
ACMG criterion in the internal point total.

### Second-allele candidates

- `p.Asn1002Lys`: REVEL 0.472 remains in the calibrated indeterminate zone. PP3 and
  BP4 are both withheld. No NMD/hypomorph claim is made from its exon position.
- `c.2679-1026A>G`: no splice criterion is assigned. The public REST workflow did
  not produce a SpliceAI score; splice alteration remains a testable hypothesis.

### PM3 / phase

The old narrative that the case is simply "one point short of Pathogenic" is removed.
PM3 strength depends on the classification of the other allele and phase evidence.
Here the second allele remains unresolved and trans is not established, so PM3 is not
used to manufacture certainty about the biallelic genotype.


### 8a · REVEL against the Pejaver calibration

The classification below does not hard-code a REVEL value. The band each variant
falls into is derived from the VEP response, so re-running the notebook
re-derives the decision to apply or withhold PP3/BP4 rather than restating it.

The ladder itself is defined **once, in section 6**, before candidate triage uses
it. This section applies and displays that single definition rather than
repeating the cut-offs — so triage and interpretation can never disagree about
where a threshold sits.

Thresholds are the **full Pejaver et al. (2022) ladder**, not just the two
Supporting cut-offs. REVEL is calibrated at several evidence strengths on each
side, and a variant qualifies at the strongest level its score earns:

| REVEL | Criterion |
|---|---|
| >= 0.932 | PP3_Strong |
| >= 0.773 | PP3_Moderate |
| >= 0.644 | PP3_Supporting |
| 0.290 - 0.644 | **indeterminate — no criterion at any strength** |
| <= 0.290 | BP4_Supporting |
| <= 0.053 | BP4_Moderate |
| <= 0.016 | BP4_Strong |
| <= 0.003 | BP4_VeryStrong (capped at Strong in practice) |

This matters for both decisions the analysis makes. `p.Asn1002Lys` at 0.472 sits
in the indeterminate band — it fails not only PP3_Supporting but every strength
on both sides, which is a stronger statement than "below the PP3 cut-off".
`SGO1 p.Thr425Ala` at 0.011 clears BP4_Strong, not merely BP4_Supporting, which
is why it is dropped rather than carried as a weak candidate.

In [ ]:
# The ladder and revel_band() are defined once, in section 6, so that candidate
# triage consults the same thresholds this section reports. Nothing is redefined
# here - this cell only applies and displays them.
assert "REVEL_PATHOGENIC" in globals() and "revel_band" in globals(), \
    "run section 6 first: the Pejaver ladder is defined there"

revel = {}
for v in ann:
    for tc in v.get("transcript_consequences", []):
        if tc.get("canonical"):
            revel[v.get("input")] = parse_revel(tc.get("revel_score"))

print("=== REVEL vs PEJAVER CALIBRATION (full threshold ladder) ===")
for key, score in revel.items():
    crit, strength, why = revel_band(score)
    print(f"  {key}")
    print(f"    -> {crit or 'none'}: {why}")

print("\nLadder in use:")
for cutoff, crit, _ in REVEL_PATHOGENIC:
    print(f"  REVEL >= {cutoff:<5} -> {crit}")
print(f"  {BP4_MAX} < REVEL < {PP3_MIN}  -> indeterminate, no criterion")
for cutoff, crit, _ in reversed(REVEL_BENIGN):
    print(f"  REVEL <= {cutoff:<5} -> {crit}")

In [ ]:
POINTS = {"PVS": 8, "PS": 4, "PM": 2, "PP": 1,
          "BP": -1, "BM": -2, "BS": -4, "BA": -8}

def classify_acmg(criteria):
    pts = sum(POINTS[s] for _, s in criteria)
    if pts >= 10:   cls = "Pathogenic"
    elif pts >= 6:  cls = "Likely Pathogenic"
    elif pts >= 0:  cls = "VUS"
    elif pts >= -6: cls = "Likely Benign"
    else:           cls = "Benign"
    return pts, cls


def gnomad_af_for(pos, frame):
    x = frame[frame.pos == pos]
    if not len(x):
        return np.nan
    return x.gnomad_max_af.iloc[0]


def pm2_if_absent(pos, frame):
    af = gnomad_af_for(pos, frame)
    return [("PM2", "PP")] if pd.isna(af) else []

ASSESSMENTS = {
    "BUB1B c.2210T>G p.Leu737Ter": {
        # PM2 intentionally withheld: current ClinVar record warns that population
        # frequency data at this position are unreliable.
        "criteria": [("PVS1", "PVS")],
        "rationale": (
            "PVS1: nonsense/LoF in BUB1B, with NMD predicted. Population-frequency "
            "evidence is not load-bearing because ClinVar flags gnomAD quality at "
            "this position as unreliable. External evidence: NCBI ClinVar classifies "
            "the exact variant as Pathogenic for MVA1 (Variation ID 533901)."),
    },
    "BUB1B c.3006T>G p.Asn1002Lys": {
        "criteria": pm2_if_absent(40220612, ann_df),
        "rationale": (
            "PP3/BP4 withheld: REVEL 0.472 is indeterminate under the Pejaver "
            "calibration. No NMD or hypomorphic inference is made from exon position."),
    },
    "BUB1B c.2679-1026A>G": {
        "criteria": pm2_if_absent(40216470, deep_ann_df),
        "rationale": (
            "No splice criterion applied. The public Ensembl REST annotation used "
            "here did not return a SpliceAI plugin score; a pseudo-exon effect remains "
            "a hypothesis requiring dedicated prediction and RNA/minigene evidence."),
    },
}

# Robustness of the PM2 decision on the lead allele.
#
# PM2_Supporting was dropped for p.Leu737Ter because the ClinVar submitter flags
# gnomAD quality at this position as unreliable. Relying on a third party's claim
# about data quality would be weak grounds on its own, so the decision is tested
# rather than asserted: the classification is computed both ways.
lead_with_pm2 = classify_acmg([("PVS1", "PVS"), ("PM2", "PP")])
lead_without  = classify_acmg([("PVS1", "PVS")])
print("=== PM2 ROBUSTNESS CHECK (lead allele p.Leu737Ter) ===")
print(f"  with PM2_Supporting    : {lead_with_pm2[0]} pts -> {lead_with_pm2[1]}")
print(f"  without PM2            : {lead_without[0]} pts -> {lead_without[1]}")
print(f"  classification changes : {lead_with_pm2[1] != lead_without[1]}")
print("  -> The call does not depend on the PM2 decision, so it does not rest on")
print("     the ClinVar submitter's frequency-quality claim. PM2 is withheld as")
print("     the conservative option, not as a load-bearing one.\n")

print("=== ACMG/AMP INTERPRETATION (internal evidence kept separate from ClinVar) ===\n")
for name, a in ASSESSMENTS.items():
    pts, cls = classify_acmg(a["criteria"])
    codes = ", ".join(f"{c}_{s}" for c, s in a["criteria"]) or "none"
    print(name)
    print(f"  criteria : {codes}")
    print(f"  points   : {pts}  ->  {cls}")
    print(f"  rationale: {a['rationale']}\n")


## 9 · Submission file — candidate #1 of up to 6

The submission keeps **two competing BUB1B pairs** that share the established
`p.Leu737Ter` allele. This is deliberate: the first allele is strong, but the data do
not resolve which second allele is causal or whether either is in trans.

The wording avoids calling the genotype proven compound heterozygosity. EPCR values
are treated as **submission-ranking confidence**, not as clinical posterior
probabilities. They remain easy to tune in one place if the other five submission
slots are used to explore calibration.

**EPCR calibration.** The two rows are mutually exclusive hypotheses about the same
second allele, so their probabilities must sum to no more than 1 and the remainder is
the probability that neither is right. That remainder is not zero: no CNV/SV caller
was run, the FASTQ were never realigned, and no splice prediction was obtained.

- coding + coding hypothesis: **0.60**
- coding + deep-intronic hypothesis: **0.25**
- residual mass for "the second allele is something else": **0.15**

The values are computed and asserted in the cell below rather than restated here, so
prose and code cannot drift apart.


In [ ]:
PROBAND_ID = SAMPLE
TEAM = "fernandosr85"
# EPCR calibration.
#
# The two rows are MUTUALLY EXCLUSIVE hypotheses about the same second allele: at
# most one can be the causal partner of the established nonsense allele. Their
# probabilities must therefore sum to no more than 1, and the remainder is the
# probability that neither is correct.
#
# That remainder is not zero. No CNV/SV caller was run, the FASTQ were never
# realigned, no splice prediction was obtained, and the corrected gnomAD-first
# shortlist may yet surface a candidate the old dbSNP gate hid. Reserving explicit
# mass for "neither" is the honest expression of that, and F-max scores calibration
# rather than confidence.
PRIMARY_EPCR = 0.60          # nonsense + p.Asn1002Lys
DEEP_INTRONIC_EPCR = 0.25    # nonsense + c.2679-1026A>G

RESIDUAL = round(1.0 - PRIMARY_EPCR - DEEP_INTRONIC_EPCR, 3)
assert RESIDUAL >= 0, "mutually exclusive hypotheses cannot sum above 1.0"
print(f"EPCR mass on stated hypotheses: {PRIMARY_EPCR + DEEP_INTRONIC_EPCR:.2f}")
print(f"Residual mass for 'second allele is something else': {RESIDUAL:.2f}")
print("  (uncalled variant, CNV/SV, regulatory, or a locus outside the panel)\n")

def row(v1, v2, epcr, finding_type, notes):
    """One Track 1 row. v = (chrom, pos, ref, alt), without a 'chr' prefix."""
    c1, p1, r1, a1 = v1
    out = {"proband_id": PROBAND_ID,
           "chrom_1": f"chr{c1}", "pos_1": int(p1), "ref_1": r1, "alt_1": a1,
           "chrom_2": "", "pos_2": "", "ref_2": "", "alt_2": "",
           "epcr": epcr, "finding_type": finding_type, "notes": notes}
    if v2:
        c2, p2, r2, a2 = v2
        out.update({"chrom_2": f"chr{c2}", "pos_2": int(p2),
                    "ref_2": r2, "alt_2": a2})
    return out

NONSENSE = ("15", 40209701, "T", "G")
MISSENSE = ("15", 40220612, "T", "G")
INTRONIC = ("15", 40216470, "A", "G")

sub = pd.DataFrame([
    row(NONSENSE, MISSENSE, PRIMARY_EPCR, "primary",
        "Candidate biallelic BUB1B/MVA1 genotype. c.2210T>G p.Leu737Ter is an "
        "established pathogenic ClinVar variant for MVA1 (rs759242053; Variation "
        "ID 533901) and independently supports PVS1. Second allele c.3006T>G "
        "p.Asn1002Lys remains VUS; REVEL 0.472 is indeterminate, so PP3/BP4 are "
        "withheld. The two coding variants are 10.9 kb apart with no shared PID; "
        "trans is not demonstrated. Allele balance is compatible with constitutional "
        "heterozygosity but does not prove germline origin."),
    row(NONSENSE, INTRONIC, DEEP_INTRONIC_EPCR, "primary",
        "Alternative BUB1B second-allele hypothesis. c.2679-1026A>G is deep "
        "intronic and prioritized as a possible splice-altering variant, but no "
        "splice criterion is assigned: the public Ensembl REST workflow did not "
        "provide a SpliceAI plugin score. Pseudo-exon activation is untested and "
        "would require dedicated splice prediction plus RNA/minigene validation. "
        "Phase with p.Leu737Ter is unresolved."),
])

path = f"/kaggle/working/{TEAM}_bub1b-panel-acmg_audited.csv"
sub.to_csv(path, index=False)
print(sub.to_string(index=False), "\n")
print("written:", path)


## 10 · Cleanup

DELETE-list material remains the raw WGS files and broad per-variant genotype tables,
including `panel_variants.tsv`. The notebook intentionally carries no saved outputs
containing those tables.

KEEP-list material used for the public repo is limited to code, report, ranked
submission findings, HPO terms and the small named-variant annotation artefacts:
`annotated_candidates.csv` — the corrected shortlist, each variant carrying an
explicit disposition and the reason for it, plus any variant the old dbSNP-gated
filter admitted that the corrected filter drops — and `deep_intronic_candidate.csv`
(one named hypothesis). These carry annotation and disposition, never genotype, depth,
allele fraction or phase fields, so they are findings rather than a genotype table.

Run deletion only when hackathon work is finished. The Official Rules require deletion
**within 30 days of the hackathon close**, across every environment you control —
local machines, cloud instances, notebooks, private repositories, and any intermediate
or derived dataset — followed by an attestation email to
`RarediseaserealkidMVAhackathon2026@synapse.org`.

**Embargo.** Code, models and results may be shared publicly at any time. A
peer-reviewed manuscript using this dataset may not be submitted until the organisers
publish their summary report or preprint. Conference abstracts and posters need prior
written approval.

**Before committing the notebook to a public repository:** clear all cell outputs. The
executed outputs contain per-variant genotype tables, which are DELETE-list material,
and the repository becomes public after the hackathon closes.


In [ ]:
import shutil

CONFIRM_DELETE = False        # set True only when the hackathon work is finished

if CONFIRM_DELETE:
    shutil.rmtree(DEST, ignore_errors=True)
    for f in ["panel_variants.tsv"]:
        p = f"/kaggle/working/{f}"
        if os.path.exists(p):
            os.remove(p)
    print("controlled data removed from", DEST)
    print()
    print("Next, per the Official Rules:")
    print("  email RarediseaserealkidMVAhackathon2026@synapse.org to attest deletion")
    print("  deadline: within 30 days of the hackathon close")
    print("  state the systems you control that were cleared, and name the provider")
    print("  and data-handling configuration of every external tool used")
else:
    print("cleanup disabled. Set CONFIRM_DELETE=True when finished.")
    print(f"controlled data currently under: {DEST}")